In [6]:
print("hello")

hello


In [7]:

import argparse, json, os, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO


In [8]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
COCO_DIR    = Path("./coco")
VAL_IMG_DIR = COCO_DIR / "val2017" 
ANN_FILE    = COCO_DIR / "annotations/instances_val2017.json"
IMG_SIZE    = 640
DEVICE      = "cuda"      # "cuda" | "cpu" | "mps"
CONF        = 0.001       # low → recall-friendly (standard for mAP eval)
IOU         = 0.65
WARMUP      = 3
RESULTS_CSV = "results_yolov10.csv"

# YOLOv10 introduces NMS-free head (end-to-end detection)
# Ultralytics handles it transparently via YOLO("yolov10*.pt")
MODELS = [
    "yolov10n.pt",
    "yolov10s.pt",
    "yolov10m.pt",
    "yolov10b.pt",
    "yolov10l.pt",
    "yolov10x.pt",
]

In [9]:
# ── COCO DOWNLOAD ─────────────────────────────────────────────────────────────
def download_coco():
    COCO_DIR.mkdir(parents=True, exist_ok=True)
    files = {
        "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
        "annotations_trainval2017.zip":
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    }
    for fname, url in files.items():
        dest = COCO_DIR / fname
        if not dest.exists():
            print(f"Downloading {fname} ...")
            urllib.request.urlretrieve(url, dest)
            with zipfile.ZipFile(dest) as z:
                z.extractall(COCO_DIR)
            print(f"  ✔ Extracted {fname}")


In [10]:
# ── COCO CATEGORY MAP ─────────────────────────────────────────────────────────
def build_id_map(ann_file):
    """Map 0-indexed YOLO class → COCO category_id."""
    with open(ann_file) as f:
        data = json.load(f)
    cats = sorted(data["categories"], key=lambda c: c["id"])
    return {i: c["id"] for i, c in enumerate(cats)}

In [11]:
# ── EVALUATION ────────────────────────────────────────────────────────────────
def coco_eval(coco_gt, predictions):
    if not predictions:
        return 0.0, 0.0
    coco_dt = coco_gt.loadRes(predictions)
    ev = COCOeval(coco_gt, coco_dt, "bbox")
    ev.evaluate()
    ev.accumulate()
    ev.summarize()
    return round(ev.stats[1], 4), round(ev.stats[0], 4)

In [12]:
# ── SINGLE MODEL BENCHMARK ────────────────────────────────────────────────────
def benchmark(model_name, img_ids, coco_gt, id_map):
    print(f"\n{'='*55}")
    print(f"  Model : {model_name}")
    print(f"{'='*55}")

    model = YOLO(model_name)

    weight_path = Path(model.ckpt_path) if hasattr(model, "ckpt_path") else Path(model_name)
    size_mb  = weight_path.stat().st_size / 1e6 if weight_path.exists() else 0.0
    params_m = sum(p.numel() for p in model.model.parameters()) / 1e6

    # Warm-up
    dummy = str(next(VAL_IMG_DIR.glob("*.jpg")))
    for _ in range(WARMUP):
        model(dummy, imgsz=IMG_SIZE, device=DEVICE,
              conf=CONF, iou=IOU, verbose=False)

    predictions, latencies = [], []

    for img_id in tqdm(img_ids, desc=model_name, unit="img"):
        img_path = VAL_IMG_DIR / f"{img_id:012d}.jpg"
        if not img_path.exists():
            continue

        t0 = time.perf_counter()
        results = model(str(img_path), imgsz=IMG_SIZE, device=DEVICE,
                        conf=CONF, iou=IOU, verbose=False)[0]
        latencies.append((time.perf_counter() - t0) * 1000)

        boxes  = results.boxes.xyxy.cpu().numpy()
        scores = results.boxes.conf.cpu().numpy()
        cls    = results.boxes.cls.cpu().numpy().astype(int)

        for box, score, c in zip(boxes, scores, cls):
            x1, y1, x2, y2 = box
            predictions.append({
                "image_id":    img_id,
                "category_id": id_map.get(c, c + 1),
                "bbox":        [float(x1), float(y1),
                                float(x2 - x1), float(y2 - y1)],
                "score":       float(score),
            })

    map50, map5095 = coco_eval(coco_gt, predictions)
    avg_ms = round(float(np.mean(latencies)), 2)

    row = {
        "model":         model_name.replace(".pt", ""),
        "mAP@0.5":       map50,
        "mAP@0.5:0.95":  map5095,
        "speed_ms/img":  avg_ms,
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }
    print(f"\n  ▶ mAP@0.5        : {map50}")
    print(f"  ▶ mAP@0.5:0.95   : {map5095}")
    print(f"  ▶ Speed (ms/img)  : {avg_ms}")
    print(f"  ▶ Size (MB)       : {row['size_MB']}")
    print(f"  ▶ Params (M)      : {row['params_M']}")
    del model
    return row


In [13]:
# ── MAIN ──────────────────────────────────────────────────────────────────────
import traceback
import gc
import torch


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--quick", action="store_true",
                        help="Run on 500 images instead of full 5k")
    args, _ = parser.parse_known_args() 

    download_coco()

    coco_gt = COCO(str(ANN_FILE))
    img_ids = coco_gt.getImgIds()
    if args.quick:
        img_ids = img_ids[:500]
        print(f"[Quick mode] Using {len(img_ids)} images.")

    id_map = build_id_map(ANN_FILE)
    rows   = []

    for model_name in MODELS:
        try:
            row = benchmark(model_name, img_ids, coco_gt, id_map)
            rows.append(row)
        except Exception as e:
            print(f"  ⚠ Skipped {model_name}: {e}")
            traceback.print_exc()
        finally:                               # ← ADDED: runs after every model
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            print(f"  🧹 VRAM freed — "
                f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | "
                f"Cached: {torch.cuda.memory_reserved()/1e9:.2f} GB")

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)

    print(f"\n{'='*55}")
    print("  YOLOv10 — Final Results")
    print(f"{'='*55}")
    print(df.to_string(index=False))
    print(f"\n✅ Saved → {RESULTS_CSV}")

if __name__ == "__main__":
    main()


loading annotations into memory...
Done (t=0.58s)
creating index...
index created!

  Model : yolov10n.pt


yolov10n.pt: 100%|██████████| 5000/5000 [01:29<00:00, 55.57img/s]


Loading and preparing results...
DONE (t=1.68s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=38.15s).
Accumulating evaluation results...
DONE (t=6.98s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.382
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.533
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.415
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.189
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.420
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.547
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.325
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.539
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.602
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

yolov10s.pt: 100%|██████████| 5000/5000 [03:24<00:00, 24.41img/s]


Loading and preparing results...
DONE (t=3.50s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=138.37s).
Accumulating evaluation results...
DONE (t=24.59s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.459
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.627
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.500
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.265
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.506
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.630
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.357
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.594
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.654
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets

...

Loading and preparing results...
DONE (t=2.61s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=66.29s).
Accumulating evaluation results...
DONE (t=11.38s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.520
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.689
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.568
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.348
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.574
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.680
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.391
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.645
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.701
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

yolov10l.pt: 100%|██████████| 5000/5000 [04:44<00:00, 17.59img/s]


Loading and preparing results...
DONE (t=1.88s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=67.67s).
Accumulating evaluation results...
DONE (t=12.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.525
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.694
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.573
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.352
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.579
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.691
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.392
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.651
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.707
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

yolov10x.pt: 100%|██████████| 5000/5000 [05:31<00:00, 15.08img/s]


Loading and preparing results...
DONE (t=2.20s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=68.30s).
Accumulating evaluation results...
DONE (t=11.25s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.538
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.706
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.586
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.366
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.593
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.703
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.396
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.661
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.715
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

In [14]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(RESULTS_CSV)

# Style the table
styled = (
    df.style
    .set_caption("YOLOv10 — Benchmark Results on COCO val2017")
    .format({
        "mAP@0.5":      "{:.4f}",
        "mAP@0.5:0.95": "{:.4f}",
        "speed_ms/img": "{:.1f} ms",
        "size_MB":      "{:.1f} MB",
        "params_M":     "{:.1f} M",
    })
    .highlight_max(subset=["mAP@0.5", "mAP@0.5:0.95"], color="black")
    .highlight_min(subset=["speed_ms/img", "size_MB", "params_M"], color="black")
    .set_properties(**{"text-align": "center", "font-size": "13px"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "15px"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "th",
         "props": [("background-color", "#000000"), ("color", "white"),
                   ("font-size", "13px"), ("text-align", "center"), ("padding", "8px")]},
        {"selector": "tr:hover",
         "props": [("background-color", "#0B0B0B")]},
    ])
    .hide(axis="index")
)

display(styled)

model,mAP@0.5,mAP@0.5:0.95,speed_ms/img,size_MB,params_M
yolov10n,0.5327,0.3822,16.9 ms,5.9 MB,2.8 M
yolov10s,0.6267,0.4592,38.7 ms,16.6 MB,8.1 M
yolov10m,0.6754,0.5057,75.9 ms,33.6 MB,16.6 M
yolov10b,0.6888,0.5199,45.6 ms,41.7 MB,20.6 M
yolov10l,0.6943,0.5254,54.7 ms,52.4 MB,25.9 M
yolov10x,0.7063,0.5382,64.3 ms,64.4 MB,31.8 M
